# GROMACS SAXS INSTRUCTION #

## Step 1
Install the saxs-version gromacs to local/cluster machine

https://gitlab.com/cbjh/gromacs-saxs/-/wikis/Clone-and-run

In [ ]:
module purge
module load all
module load gcc cuda/10.0

source /home/wh1165/saxs/gromacs-saxs/build/dist/bin/GMXRC

## Step 2

### Run two simulations:

If the original MD data is too large, extract only a subset using the -dt option, such as every 100/1000 frames.
Alternatively, convert a single .gro or .pdb file into an .xtc file to analyze just one frame.

(If the ultimate goal of the MD simulation is to compute SAXS, then when building the initial model, the box dimensions must be sufficiently large—use the -d option, e.g., 1.5 nm—to ensure that the final scattering envelope does not exceed the box boundaries.)

In [ ]:
1. one of the molecule(s) of interest in solution (Usually, it has been done when you run normal MD)
2. one with solvent only (e.g., water + ions but no protein/DNA/RNA).

#### Notes

In [ ]:
Select and save an appropriate subset of frames from both simulations.

The number of frames from solution and solvent simulations should be similar.

Because it takes a long time to process each frame.

## Step 3

#### Make a solution folder:

In [ ]:
1. Put there the original topol.top, md.xtc, index.ndx and solution.pdb (reference structure, can be ectracted
                                                                         from solution.xtc) file

[renaming files: topol.top>>>saxs-topol.top, md.xtc>>>>solution.xtc, index.ndx, solution.pdb]

2. gromacs-saxs cannot recognize .tpr files generated by newer versions of GROMACS. Therefore, when generating trajectory-based files (e.g., solution.pdb) from simulations run with a newer GROMACS version, you must load the same newer version of GROMACS to ensure compatibility.

#### Make a solvent folder:

In [ ]:
1. Analyze the traj of solution.xtc, calculate the ion bulk concentration of the solution box.

2. Generate a water box with same dimension, add cations and ions to build a solvent box
   with same ion strength.

3. Run a short solvent MD (~20ns)

## Step 4

#### Cromer-Mann Parameters

Copy file cromer_mann_defs.itp from GROMACS installation folder (CHARMM27) to yor local folder. (可能是在CHARMM27的外面)

### Go to solution folder

To make sure that the chemical elements are correctly identified, we use atomic masses as available in a tpr. <<dummy.tpr>>

In [ ]:
touch empty.mdp
gmx grompp -f empty.mdp -c solution.pdb -p saxs-topol.top -o dummy.tpr

solution.pdb and saxs-topol.top from step 3

#### Generate a Cromer-Mann parameters file for your molecule of interest，select RNA in this case

In [ ]:
gmx genscatt -s dummy.tpr -vsites -nsl
; Remove lines of NSL if you are not using neutron scattering lengths

In [ ]:
A file named scatter_<name-of-molecule, your selection>.itp will be generated
Make sure that no line reads CROMER_MANN_UNKNOWN

In [ ]:
Revise the resulting itp file,
Replace each CROMER_MANN_<element> macro definition with explicit parameters from the CHARMM27 file
---cromer_mann_defs.itp---

Command: sed -i -e 's/CROMER_MANN_H/XXX XXX XXX XXX.../g' scatter_<name-of-molecule, your selection>.itp

Example:

[ scattering_params ]
; atom ft a1     a2      a3      a4      b1      b2      b3      b4      c
    1  1   CROMER_MANN_H    ;  H5T -    C-1
    >>>>>>>
[ scattering_params ]
; atom ft a1     a2      a3      a4      b1      b2      b3      b4      c
    1  1  0.49300   0.32291   0.14019   0.04081  10.51090  26.12570   3.14236  57.79970   0.00304  ;  H5T -    C-1
    <<<<<<<

In [ ]:
Remove lines of NSL if you are not using neutron scattering lengths

#### Create a file called “scatter_water.itp”

or directly edit ff/tip3p.itp file

In [ ]:
Just like what we have done for scatter_<name-of-molecule, your selection>.itp:
Take tip3p water as example. The second column "1" mean these three atoms (O H H) belong to the same molecule.

 [ scattering_params ]
; atom ft a1     a2      a3      a4      b1      b2      b3      b4      c
1 1 1.74364   2.56338   3.42821   0.95684   0.33326   5.70713  13.32240  33.18740   0.26793   ;CROMER_MANN_Owt
2 1 0.07852   0.26770   0.16044   0.01140   3.20156  10.91840  28.25590  69.45850   0.00195
3 1 0.07852   0.26770   0.16044   0.01140   3.20156  10.91840  28.25590  69.45850   0.00195

#### Edit ion.itp in your force field folder, if there are ions in system

In [ ]:
Just like what we have done for scatter_water.itp:
Take CL ions as example, <<<CL_minus>>>

[ moleculetype ]
; molname       nrexcl
CL              1

[ atoms ]
; id    at type         res nr  residu name     at name  cg nr  charge
   1    Cl                  1       CL             CL      1   -1.00000

[ scattering_params ]     ; added
; atom ft a1     a2      a3      a4      b1      b2      b3      b4      c
1  1  18.29150   7.20840   6.53370   2.33860   0.00660   1.17170  19.54240  60.44860 -16.37800    ;CL_minus

#### Edit .top files and include the .itp files needed in saxs calculation

In [ ]:
...
#ifdef POSRES_WATER
; Position restraint for each water oxygen
[ position_restraints ]
;  i funct       fcx        fcy        fcz
   1    1       1000       1000       1000
#endif
...

If in solution folder:

; Include scattering topology
#include "scatter_<name-of-molecule, your selection>.itp"          <<<<< her
; Include scattering topology
#include "scatter_water.itp"                                       <<<<< here

If in solvent folder:

; Include scattering topology
#include "scatter_water.itp"                                       <<<<< here

In [ ]:
The Cromer-Mann parameters of ions have been added to ions.itp.

## Step 5

#### Building the envelope

### Go to solution folder

1. It will:

create a structure reference file (gro/pdb) that includes ONLY the molecule of interest

generate a trajectory file (xtc) that includes ONLY the molecule of interest

In [ ]:
gmx trjconv -f solution.xtc -n index.ndx -s dummy.tpr -dump <time-frame> -o <molecule-name>.pdb
gmx trjconv -f solution.xtc -n index.ndx -s dummy.tpr -o <molecule-name>.xtc

2. Before generating the envelope:

center the molecule in the box prior to this step because the envelope is not supposed to cross the pbc box. (Depends)

In HJH case, it is fixed.

In [ ]:
gmx genenv -d 1.0 -f <molecule-name>.xtc -s <molecule-name>.pdb
;flag -d controls the size of the envelope as the distance from the surface of the molecule (typically 1 nm).

## Step 6

#### Prepare .mdp file for both of solution and solvent.

In [ ]:
make groups RNA (for solute) and Water_and_ions (for solvent), and specify in rerun.mdp:

In [ ]:
Check rerun.mdp file

关注：
Take care
define                   = -DSCATTER

; Scattering coupling stuff: xray and/or neutron (multiple neutron possible)
scatt-coupl              = xray
; Selection of solute, solvent and fit group
waxs-solute              = <name of solute>        ; for solution. removed when for solvent
waxs-solvent             = <name of solvent>
waxs-rotfit              =
; WAXS pbc atom list near center of solute (uses global indices, 0 = number-wise center, -1 = used atomic distances)
waxs-pbcatom             = <enter Global atom number from step 5 of building the envelope> ;

In [ ]:
Prepare rerun.mdp and solvent.mdp

## Step 7

###  Go to solution folder

In [ ]:
gmx grompp -f rerun.mdp -p saxs-topol.top -c solution.pdb -n index.ndx -o saxs.tpr

### Go to solvent folder

In [ ]:
gmx grompp -c solvent.pdb -f solvent.mdp -p solvent-topol.top -o solvent.tpr

### Run scattering

In [ ]:
export GMX_WAXS_FIT_REFFILE=envelope-ref.gro  ;from solution folder
export GMX_ENVELOPE_FILE=envelope.dat         ;from solution folder

gmx mdrun -s saxs.tpr -rerun solution.xtc -sw solvent.tpr -fw solvent.xtc

In [ ]:
waxs.log: contains radius of gyration from Guinier fit together with other statistics
waxs_final.xvg: I(q)

### Run GNOM

https://www.embl-hamburg.de/biosaxs/software.html

In [ ]:
datgnom4 waxs_final.xvg -r <rg from Guinier approx> -o pairwise_dist.dat

## Summary of SAXS-Driven Simulations

### General Recommendations

- **Disable rotation and motion restraints**, or the structure may drift outside the simulation box.
- **Avoid using angular rotation removal** for DNA/RNA, as it may cause the structure to break near the center of mass.  
  Instead, use a combination of **linear enforced rotation** restraints.
- If unsure about the target, **compute WAXS from a known trajectory and fit it to experimental data** to get a rough target profile.
- For **high-precision experimental data**, use **linear fitting**.  
  Logarithmic fitting often fails due to negative offsets.
- For **low-precision experimental data**, logarithmic fitting may be necessary due to noise in the high-q region.
- The choice between linear and logarithmic fitting can be guided by **preliminary WAXS fitting and comparison**.

---

### Practical Issues and Observations

1. **MPI execution often causes segmentation faults**, but the cause is unclear.
2. Related to point 1: segmentation faults occur consistently when using **ensemble mode**.
3. **Observation 1**: The selected q-range influences the MD adjustment direction; the algorithm tends to prioritize fitting the high-q region.
4. **Observation 2**: Over-adjustment is common and can trigger rollback attempts, which often fail (e.g., due to negative corrections).
5. Running on **NVIDIA nodes** can cause errors such as the structure escaping a compact box.
6. The `waxs-tau` parameter adjusts the **fitting rate**.  
   A value of `250` is generally appropriate, while `500` results in slower convergence.
7. Without **positional restraints (`posre`)**, errors such as the structure escaping the box are frequent.
8. Using **scale-and-offset fitting** may create discrepancies between the machine-learned (ML) target and the true experimental target.
9. It's advisable to **scale the target curve**, e.g., by multiplying by `380000`.  
   However, directly using experimental data as the target often leads to structural degradation.
10. **Always run a short initial MD simulation** before starting SAXS-driven fitting to derive an ML-compatible target profile for comparison.
